In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
def altitud_ingreso(ingreso):
    if ingreso is None:
        return "No Registrado"
    elif ingreso < 2500:
        return "Bajo"
    elif 2500 <= ingreso < 7000:
        return "Medio"
    else:
        return "Alto"

In [0]:
ingreso_udf = F.udf(altitud_ingreso, StringType())

In [0]:
df_clientes = spark.table(f"{catalogo}.{esquema_source}.clientes")
df_transacciones = spark.table(f"{catalogo}.{esquema_source}.transacciones")

In [0]:
df_clientes_clean = df_clientes.dropna(how="all") \
    .filter(col("cliente_id").isNotNull()) \
    .withColumn("score_riesgo", regexp_replace(col("score_riesgo").cast(StringType()), ",", ".")) \
    .withColumn("score_riesgo", regexp_replace(col("score_riesgo"), "%", "").cast(DoubleType())) \
    .withColumn("ingreso_category", ingreso_udf(col("ingreso_mensual"))) \
    .withColumn("banco_principal_clean", trim(col("banco_principal")))

df_tx_clean = df_transacciones.dropna(how="all") \
    .filter((col("transaccion_id").isNotNull()) | (col("cliente_id").isNotNull())) \
    .withColumn("monto_operacion_round", round(col("monto_operacion"), 2)) \
    .withColumn("total_canales_usados", col("uso_cajeros") + col("uso_agencias") + col("uso_pos")) \
    .withColumn("flag_dispositivo_sospechoso", when((col("dispositivo_nuevo") == 1) & (col("fallas_autenticacion") > 1), lit("Alerta")).otherwise(lit("OK")))

In [0]:
#### df_clientes_clean = df_clientes_clean.withColumn("ingreso_category", ingreso_udf("ingreso_mensual"))

In [0]:
df_joined = df_tx_clean.alias("x").join(
    df_clientes_clean.alias("y"), 
    col("x.cliente_id") == col("y.cliente_id"), 
    "left"
)

In [0]:
df_filtered_sorted = df_joined.filter(col("monto_operacion") > 0).orderBy("transaccion_id")

In [0]:
df_filtered_sorted = df_filtered_sorted.withColumn(
    "years_diferences", 
    F.year(F.current_date()) - F.year(F.current_date())
)

In [0]:
df_aggregated = df_filtered_sorted.groupBy("departamento", "banco_principal").agg(
    F.count("transaccion_id").alias("num_transacciones")
)

In [0]:
df_with_latitude_diff = df_filtered_sorted.withColumn(
    "lat_diff", 
    F.abs(df_filtered_sorted["monto_operacion"] - df_filtered_sorted["monto_promedio_transaccion"]).cast(IntegerType())
)

In [0]:
df_updated = df_with_latitude_diff.select(
    "*",
    when(col("banco_principal").isin("BCP", "BBVA", "Interbank"), lit("Internacional")).otherwise(lit("Local")).alias("race_type"),
    when((col("score_riesgo") > 0) & (col("score_riesgo") < 50), lit("Cerca del ecuador")).otherwise(lit("Lejos del ecuador")).alias("near_equator")
).drop(col("y.cliente_id"), col("y._fecha_ingesta"))

In [0]:
df_tx_transformed = df_tx_clean \
    .withColumn("monto_operacion_round", round(col("monto_operacion"), 2)) \
    .withColumn("total_canales_usados", col("uso_cajeros") + col("uso_agencias") + col("uso_pos")) \
    .withColumn("flag_dispositivo_sospechoso", when((col("dispositivo_nuevo") == 1) & (col("fallas_autenticacion") > 1), lit("Alerta")).otherwise(lit("OK")))

In [0]:
# JOIN en PySpark por la llave primaria/foránea
df_joined = df_tx_transformed.alias("x").join(
    df_clientes_clean.alias("y"),
    col("x.cliente_id") == col("y.cliente_id"),
    "left"
)

In [0]:
df_silver_final = df_joined.select(
    col("x.*"),
    col("y.edad"),
    col("y.ingreso_mensual"),
    col("y.antiguedad_bancaria"),
    col("y.departamento"),
    col("y.banco_principal"),
    col("y.app_preferida"),
    col("y.tipo_operacion_frecuente"),
    col("y.monto_promedio_transaccion"),
    col("y.tiene_token_fisico"),
    col("y.tiene_token_digital"),
    col("y.score_riesgo"), # <--- PROYECCIÓN EXPLÍCITA
    col("y.fecha_registro")
).withColumn(
    "diferencia_monto_promedio", 
    abs(col("monto_operacion_round") - col("monto_promedio_transaccion"))
).withColumn(
    "monto_excede_ingreso",
    when(col("monto_operacion") > col("ingreso_mensual"), lit(True)).otherwise(lit(False))
).withColumn(
    "nivel_riesgo",
    when((coalesce(col("score_riesgo"), lit(0)) > 0.7) & (col("ubicacion_inusual") == 1), lit("Alto"))
    .when((coalesce(col("score_riesgo"), lit(0)) > 0.4), lit("Medio"))
    .otherwise(lit("Bajo"))
).withColumn(
    "ingestion_date_silver", current_timestamp()
)

In [0]:
df_silver_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalogo}.{esquema_sink}.transacciones_clientes_silver")

print(" Transformación a Capa Silver completada exitosamente guardando df_silver_final.")

In [0]:
spark.table("catalog_au.silver.transacciones_clientes_silver").count()